#  Imports e Configuração de Caminhos

In [ ]:
# ============================================================
# CÉLULA 1 — Imports e Configuração (adaptado para VSCode/local)
# ============================================================
import os
import sys
import random
import shutil
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

import ultralytics
print(f"Ultralytics versão: {ultralytics.__version__}")
print(f"OpenCV versão:      {cv2.__version__}")
print(f"Python:             {sys.version}")

# ─── CAMINHOS BASE (ajuste conforme sua máquina) ──────────────
PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
DATA_RAW     = PROJECT_ROOT / "data" / "wood_raw"
WORK         = PROJECT_ROOT / "dataset" / "wood_yolo"
RUNS_DIR     = PROJECT_ROOT / "runs"
MODELS_DIR   = PROJECT_ROOT / "models"

for d in [DATA_RAW, WORK, RUNS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── CONFIGURAÇÃO GERAL ───────────────────────────────────────
CFG = dict(
    imgsz       = 640,
    epochs      = 100,
    batch       = 16,           # Reduza para 4 se der OOM na GPU
    workers     = 4,           # Windows: use 0 se der erro de DataLoader
    val_size    = 0.2,
    seed        = 42,
    model       = "yolov8s.pt",
    min_box_px  = 4,
    conf        = 0.30,
    iou         = 0.55,
)
SEED = CFG["seed"]
random.seed(SEED)
np.random.seed(SEED)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

print("✅ Configuração carregada!")
print(f"   PROJECT_ROOT : {PROJECT_ROOT}")
print(f"   DATA_RAW     : {DATA_RAW}")
print(f"   WORK         : {WORK}")

c:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Ultralytics versão: 8.3.253
OpenCV versão:      4.13.0
Python:             3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
✅ Configuração carregada!
   PROJECT_ROOT : c:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026
   DATA_RAW     : c:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\data\wood_raw
   WORK         : c:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\dataset\wood_yolo


# Download do Dataset via Kaggle CLI

In [2]:
# ============================================================
# CÉLULA 2 — Download do Dataset (local, via kaggle CLI)
#
# Pré-requisito: coloque o arquivo kaggle.json em:
#   Windows : C:\Users\<Usuário>\.kaggle\kaggle.json
#   Linux   : ~/.kaggle/kaggle.json
#   Mac     : ~/.kaggle/kaggle.json
#
# Obtenha em: https://www.kaggle.com/settings → API → Create New Token
# ============================================================
import subprocess
import platform

KAGGLE_DATASET = "nomihsa965/large-scale-image-dataset-of-wood-surface-defects"

# Verifica se o kaggle.json existe
kaggle_cfg = Path.home() / ".kaggle" / "kaggle.json"
if not kaggle_cfg.exists():
    raise FileNotFoundError(
        f"Arquivo kaggle.json não encontrado em {kaggle_cfg}.\n"
        "Baixe em: https://www.kaggle.com/settings → API → Create New Token"
    )

# Verifica se já foi extraído
already_extracted = list(DATA_RAW.rglob("*.jpg"))
if len(already_extracted) > 100:
    print(f"✅ Dataset já extraído ({len(already_extracted)} imagens encontradas). Pulando download.")
else:
    print("⏬ Baixando dataset do Kaggle... (pode demorar dependendo da sua internet)")
    result = subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
         "--unzip", "-p", str(DATA_RAW)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError("Falha no download. Verifique suas credenciais do Kaggle.")
    
    imgs = list(DATA_RAW.rglob("*.jpg"))
    print(f"✅ Download concluído! {len(imgs)} imagens encontradas.")

# Lista estrutura de pastas
print("\n📁 Estrutura do dataset:")
for p in sorted(DATA_RAW.iterdir()):
    if p.is_dir():
        n = len(list(p.rglob("*")))
        print(f"   {p.name}/  ({n} arquivos)")

✅ Dataset já extraído (4000 imagens encontradas). Pulando download.

📁 Estrutura do dataset:
   Bounding Boxes - YOLO Format - 1/  (4001 arquivos)
   Images - 1/  (4001 arquivos)


# Funções Utilitárias e Mapeamento

In [3]:
# ============================================================
# CÉLULA 3 — Funções auxiliares e mapeamento imagem↔label
# ============================================================

def find_files(root, exts=None):
    root = Path(root)
    files = []
    for p in root.rglob("*"):
        if p.is_file():
            if exts is None or p.suffix.lower() in exts:
                files.append(p)
    return sorted(files)

def safe_imread(path):
    img = cv2.imread(str(path))
    return img

def read_yolo_txt(path):
    rows = []
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) != 5:
                    continue
                cls, xc, yc, bw, bh = parts
                rows.append((int(float(cls)), float(xc), float(yc), float(bw), float(bh)))
    except Exception as e:
        print(f"  Aviso ao ler label {path}: {e}")
    return rows

def keep_box(bw, bh, img_w, img_h, min_px=4):
    return (bw * img_w >= min_px) and (bh * img_h >= min_px) and bw > 0 and bh > 0

def infer_image_map(images_dir):
    return {p.stem: p for p in find_files(images_dir, IMG_EXTS)}

def stem_to_label_path(label_root):
    return {p.stem: p for p in find_files(label_root, {".txt"})}

# ─── Procura imagens e labels dentro do dataset ───────────────
image_map = infer_image_map(DATA_RAW)
label_map = stem_to_label_path(DATA_RAW)

matched = sorted(set(image_map.keys()) & set(label_map.keys()))

print(f"📸 Imagens encontradas : {len(image_map)}")
print(f"🏷️  Labels encontradas  : {len(label_map)}")
print(f"🔗 Pares combinados    : {len(matched)}")
print(f"   Exemplos de stems  : {matched[:5]}")

📸 Imagens encontradas : 4000
🏷️  Labels encontradas  : 4000
🔗 Pares combinados    : 4000
   Exemplos de stems  : ['100000000', '100000001', '100000002', '100000003', '100000004']


# Preparação do Dataset YOLO

In [ ]:
# ============================================================
# CÉLULA 4 — Organização em estrutura YOLO (train / val)
# ============================================================

TRAIN_IMG = WORK / "images" / "train"
VAL_IMG   = WORK / "images" / "val"
TRAIN_LBL = WORK / "labels" / "train"
VAL_LBL   = WORK / "labels" / "val"

for d in [TRAIN_IMG, VAL_IMG, TRAIN_LBL, VAL_LBL]:
    d.mkdir(parents=True, exist_ok=True)

# Verifica se já foi preparado
if len(list(TRAIN_IMG.glob("*"))) > 100:
    print("✅ Dataset já preparado. Pulando re-preparação.")
else:
    # ─── Coleta registros válidos ─────────────────────────────
    records = []
    for stem in tqdm(matched, desc="Processando pares"):
        img_path = image_map[stem]
        lbl_path = label_map.get(stem)
        img = safe_imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        cleaned = []
        dropped = 0
        if lbl_path and lbl_path.exists():
            for cls, xc, yc, bw, bh in read_yolo_txt(lbl_path):
                if keep_box(bw, bh, w, h, CFG["min_box_px"]):
                    cleaned.append((cls, xc, yc, bw, bh))   # preserva classe original
                else:
                    dropped += 1

        records.append({
            "stem": stem,
            "image_path": str(img_path),
            "label_path": str(lbl_path) if lbl_path else None,
            "has_defect": int(len(cleaned) > 0),
            "num_boxes": len(cleaned),
            "dropped_boxes": dropped,
            "cleaned_rows": cleaned,
        })

    df = pd.DataFrame(records)
    print(f"\n📊 Distribuição:")
    print(df["has_defect"].value_counts().rename({1: "com_defeito", 0: "sem_defeito"}))
    print(f"Boxes descartadas: {df['dropped_boxes'].sum()}")

    # ─── Split estratificado ──────────────────────────────────
    train_df, val_df = train_test_split(
        df, test_size=CFG["val_size"], random_state=SEED,
        stratify=df["has_defect"]
    )
    print(f"\n✅ Treino: {len(train_df)} | Validação: {len(val_df)}")

    # ─── Copia arquivos ───────────────────────────────────────
    def write_split(subset_df, img_dir, lbl_dir):
        img_dir, lbl_dir = Path(img_dir), Path(lbl_dir)
        
        # --- INICIALIZA O CLAHE ---
        # clipLimit: define o limite de contraste (2.0 a 3.0 costuma ser ideal para texturas)
        # tileGridSize: divide a imagem em blocos (8x8) para equalizar o contraste localmente
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        
        for _, row in tqdm(subset_df.iterrows(), total=len(subset_df)):
            src = Path(row["image_path"])
            
            # --- INÍCIO DO PRÉ-PROCESSAMENTO ---
            # Lê a imagem original
            img = cv2.imread(str(src))
            
            # 1. Converte para tons de cinza (1 canal)
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # 2. Aplica o filtro CLAHE na imagem em tons de cinza
            gray_clahe = clahe.apply(gray)
            
            # 3. Converte de volta para BGR (3 canais idênticos, exigência do YOLO)
            img_yolo = cv2.cvtColor(gray_clahe, cv2.COLOR_GRAY2BGR)
            
            # Salva a imagem processada no destino
            cv2.imwrite(str(img_dir / src.name), img_yolo)
            # --- FIM DO PRÉ-PROCESSAMENTO ---
            
            dst_lbl = lbl_dir / f"{row['stem']}.txt"
            with open(dst_lbl, "w", encoding="utf-8") as f:
                for cls, xc, yc, bw, bh in row["cleaned_rows"]:
                    f.write(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

    print("Copiando treino...")
    write_split(train_df, TRAIN_IMG, TRAIN_LBL)
    print("Copiando validação...")
    write_split(val_df, VAL_IMG, VAL_LBL)

print("\n✅ Dataset pronto!")
print(f"   Treino : {len(list(TRAIN_IMG.glob('*')))} imagens")
print(f"   Val    : {len(list(VAL_IMG.glob('*')))} imagens")

✅ Dataset já preparado. Pulando re-preparação.

✅ Dataset pronto!
   Treino : 3200 imagens
   Val    : 800 imagens


# Criação do data.yaml


In [ ]:
# ============================================================
# CÉLULA 5 — Arquivo de configuração data.yaml
# ============================================================

data_yaml_content = {
    "path": str(WORK.resolve()),  # caminho absoluto (importante no Windows)
    "train": "images/train",
    "val":   "images/val",
    "nc": 8,
    "names": [
        "Quartzity",       # 0
        "Live_Knot",       # 1
        "Marrow",          # 2
        "resin",           # 3
        "Dead_Knot",       # 4
        "knot_with_crack", # 5
        "Knot_missing",    # 6
        "Crack",           # 7
    ],
}


YAML_PATH = WORK / "data.yaml"
with open(YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False, allow_unicode=True)

print("✅ data.yaml criado em:", YAML_PATH)
print()
with open(YAML_PATH) as f:
    print(f.read())

✅ data.yaml criado em: c:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\dataset\wood_yolo\data.yaml

names:
- wood_defect
nc: 1
path: C:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\dataset\wood_yolo
train: images/train
val: images/val



# Treinamento YOLOv8

In [ ]:
# ============================================================
# CÉLULA 6 — Treinamento YOLOv8
#
# DICA GPU:
#   - NVIDIA: instale PyTorch com CUDA em https://pytorch.org/get-started/locally/
#     Ex: pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
#   - Verifique: import torch; print(torch.cuda.is_available())
#   - Sem GPU: o treinamento roda na CPU (muito mais lento, use yolov8n.pt)
# ============================================================
import torch
device = "0" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Dispositivo de treino: {'GPU (CUDA)' if device == '0' else 'CPU'}")
if device == "0":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

model = YOLO(CFG["model"])

# Windows: workers=0 evita bugs com multiprocessing
workers = 0 if sys.platform == "win32" else CFG["workers"]

train_results = model.train(
    data    = str(YAML_PATH),
    epochs  = CFG["epochs"],
    imgsz   = CFG["imgsz"],
    batch   = CFG["batch"],
    workers = workers,
    device  = device,
    seed    = SEED,
    patience = 15,
    optimizer = "AdamW",
    lr0     = 1e-3,
    lrf     = 0.01,
    warmup_epochs = 3,
    cos_lr  = True,
    # Augmentações
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    flipud  = 0.3,
    fliplr  = 0.5,
    mosaic  = 1.0,
    mixup   = 0.1,
    # Saída
    project = str(RUNS_DIR),
    name    = "wood_yolov8s_8classes",
    exist_ok = True,
    plots   = True,
    verbose = True,
    cache = True
)

BEST_MODEL_PATH = Path(train_results.save_dir) / "weights" / "best.pt"
print(f"\n✅ Treinamento concluído!")
print(f"🏆 Melhor modelo: {BEST_MODEL_PATH}")

# Salva cópia no diretório de modelos
shutil.copy2(BEST_MODEL_PATH, MODELS_DIR / "wood_best.pt")
print(f"💾 Cópia salva em: {MODELS_DIR / 'wood_best.pt'}")

🖥️  Dispositivo de treino: GPU (CUDA)
   GPU: NVIDIA GeForce RTX 4060
New https://pypi.org/project/ultralytics/8.4.66 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.253  Python-3.14.6 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4060, 8187MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=c:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\dataset\wood_yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask

# Validação e Métricas

In [7]:
# ============================================================
# CÉLULA 7 — Validação e visualização de métricas
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

model_eval = YOLO(str(BEST_MODEL_PATH))

val_results = model_eval.val(
    data    = str(YAML_PATH),
    imgsz   = CFG["imgsz"],
    batch   = CFG["batch"],
    conf    = CFG["conf"],
    iou     = CFG["iou"],
    device  = device,
    plots   = True,
    verbose = True,
)

print("\n📈 Métricas de Validação:")
print(f"  mAP@50    : {val_results.box.map50:.4f}")
print(f"  mAP@50-95 : {val_results.box.map:.4f}")
print(f"  Precision : {val_results.box.mp:.4f}")
print(f"  Recall    : {val_results.box.mr:.4f}")

# Exibe gráficos inline no VSCode
run_dir = Path(train_results.save_dir)
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, fname in zip(axes, ["results.png", "confusion_matrix.png", "PR_curve.png"]):
    fpath = run_dir / fname
    if fpath.exists():
        ax.imshow(mpimg.imread(str(fpath)))
        ax.set_title(fname.replace(".png", ""))
    ax.axis("off")
plt.tight_layout()
plt.savefig(str(MODELS_DIR / "training_summary.png"), dpi=150)
plt.show()
print("✅ Gráfico salvo em:", MODELS_DIR / "training_summary.png")

Ultralytics 8.3.253  Python-3.14.6 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4060, 8187MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 5359.7310.1 MB/s, size: 798.8 KB)
val: Scanning C:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\dataset\wood_yolo\labels\val.cache... 800 images, 78 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 800/800 279.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 6.2it/s 4.0s0.1s
                   all        800       1764      0.903      0.858      0.905      0.518
Speed: 0.4ms preprocess, 1.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\runs\detect\val

📈 Métricas de Validação:
  mAP@50    : 0.9050
  mAP@50-95 : 0.5183
  Precision : 0.9028
  Recall    : 0.8583


<Figure size 2000x500 with 3 Axes>

✅ Gráfico salvo em: c:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\models\training_summary.png


# Teste em Imagens de Validação

In [8]:
# ============================================================
# CÉLULA 8 — Teste em imagens do dataset de validação
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

model_infer = YOLO(str(BEST_MODEL_PATH))
val_imgs = list(VAL_IMG.glob("*"))
sample_imgs = random.sample(val_imgs, min(6, len(val_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    preds = model_infer.predict(
        source  = str(img_path),
        conf    = CFG["conf"],
        iou     = CFG["iou"],
        imgsz   = CFG["imgsz"],
        device  = device,
        verbose = False,
    )
    result = preds[0]
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    n = len(result.boxes)
    status = f"⚠️ {n} defeito(s)" if n > 0 else "✅ Sem defeitos"
    ax.imshow(annotated)
    ax.set_title(f"{img_path.name}\n{status}", fontsize=8, fontweight="bold")
    ax.axis("off")

plt.suptitle("Detecção de Defeitos em Madeira — Amostras de Validação", fontsize=13, fontweight="bold")
plt.tight_layout()
out_path = MODELS_DIR / "validation_samples.png"
plt.savefig(str(out_path), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Salvo em:", out_path)

<Figure size 1800x1000 with 6 Axes>

✅ Salvo em: c:\Users\btclo\OneDrive\Documentos\Pessoal\Projetos\Omni-Root_challenge_2026\models\validation_samples.png


# Detecção em Tempo Real com Webcam (VSCode/Desktop)

In [18]:
# ============================================================
# CÉLULA 9 — Detecção em Tempo Real com Webcam
#
# DIFERENÇA PRINCIPAL EM RELAÇÃO AO COLAB:
#   Aqui usamos cv2.imshow() nativo — uma janela real abre no
#   seu desktop. Pressione 'Q' para encerrar.
#
# ATENÇÃO no Windows:
#   Se a janela travar, execute este script em um arquivo .py
#   separado (não em notebook), pois cv2.imshow tem limitações
#   com threads do Jupyter no Windows.
# ============================================================
import time
import cv2
import numpy as np
from ultralytics import YOLO

# ─── Carrega modelo ───────────────────────────────────────────
model_cam = YOLO(str(MODELS_DIR / "wood_best.pt"))

CONF_THRESH = 0.70
IOU_THRESH  = 0.55
CAM_INDEX   = 0      # Troque para 1, 2... se tiver múltiplas câmeras
WIN_NAME    = "Wood Defect Detection — Pressione Q para sair"

cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    raise RuntimeError(
        f"Não foi possível abrir a câmera (índice {CAM_INDEX}).\n"
        "Tente trocar CAM_INDEX para 1 ou 2."
    )

cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

print(f"🎥 Câmera aberta. Resolução: "
      f"{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x"
      f"{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
print("Pressione  Q  para encerrar.")

prev_time = time.time()
frame_count = 0
fps_display = 0.0

while True:
    ret, frame = cap.read()
    if not ret:
        print("❌ Falha ao ler frame. Encerrando.")
        break

    frame_count += 1

    # ─── Inferência ──────────────────────────────────────────
    results = model_cam.predict(
        source  = frame,
        conf    = CONF_THRESH,
        iou     = IOU_THRESH,
        imgsz   = 640,
        device  = device,
        verbose = False,
    )
    result = results[0]
    annotated = result.plot(line_width=2)
    n_det = len(result.boxes)

    # ─── Cálculo de FPS ──────────────────────────────────────
    curr_time = time.time()
    if curr_time - prev_time >= 0.5:
        fps_display = frame_count / (curr_time - prev_time)
        frame_count = 0
        prev_time = curr_time

    # ─── HUD (cabeçalho com status) ───────────────────────────
    h, w = annotated.shape[:2]
    bar_color = (0, 50, 200) if n_det > 0 else (0, 160, 50)
    status_txt = f"  DEFEITO DETECTADO: {n_det}" if n_det > 0 else "  SEM DEFEITOS"
    fps_txt    = f"FPS: {fps_display:.1f}"

    # Faixa semitransparente no topo
    overlay = annotated.copy()
    cv2.rectangle(overlay, (0, 0), (w, 56), bar_color, -1)
    cv2.addWeighted(overlay, 0.55, annotated, 0.45, 0, annotated)

    cv2.putText(annotated, status_txt, (6, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(annotated, fps_txt, (w - 120, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (220, 220, 220), 2, cv2.LINE_AA)

    # Confiança das detecções (se houver)
    if n_det > 0:
        confs = result.boxes.conf.cpu().numpy()
        conf_str = "  ".join([f"{c:.2f}" for c in confs[:5]])
        cv2.putText(annotated, f"conf: {conf_str}", (6, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (220, 220, 100), 1, cv2.LINE_AA)

    # ─── Exibe janela ─────────────────────────────────────────
    cv2.imshow(WIN_NAME, annotated)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q") or key == ord("Q"):
        print("🛑 Encerrado pelo usuário.")
        break

cap.release()
cv2.destroyAllWindows()
print("✅ Câmera encerrada.")

🎥 Câmera aberta. Resolução: 1280x720
Pressione  Q  para encerrar.
❌ Falha ao ler frame. Encerrando.
✅ Câmera encerrada.


#  Carregar Modelo Salvo (Sessões Futuras)

In [12]:
# ============================================================
# CÉLULA 11 — Carregar modelo já treinado (sem re-treinar)
# Execute apenas esta célula se o modelo já estiver salvo
# ============================================================
from pathlib import Path
from ultralytics import YOLO
import torch

MODELS_DIR = Path("models")  # ajuste se necessário
MODEL_FILE  = MODELS_DIR / "wood_best.pt"

device = "0" if torch.cuda.is_available() else "cpu"

if not MODEL_FILE.exists():
    raise FileNotFoundError(
        f"Modelo não encontrado em {MODEL_FILE}.\n"
        "Execute o treinamento (Célula 6) primeiro."
    )

model_loaded = YOLO(str(MODEL_FILE))
BEST_MODEL_PATH = MODEL_FILE

print(f"✅ Modelo carregado: {MODEL_FILE}")
print(f"🖥️  Dispositivo: {'GPU' if device == '0' else 'CPU'}")
print("ℹ️  Execute a Célula 9 ou 'python webcam_detect.py' para iniciar a webcam.")

✅ Modelo carregado: models\wood_best.pt
🖥️  Dispositivo: GPU
ℹ️  Execute a Célula 9 ou 'python webcam_detect.py' para iniciar a webcam.


# Export para Edge Computing (NCNN — Raspberry Pi)

In [ ]:
# ============================================================
# CÉLULA 10 — Export do modelo para NCNN (Edge Computing)
#
# O que acontece aqui:
#   1. Parte do best.pt (PyTorch, roda na GPU do seu PC)
#   2. Converte para NCNN — formato otimizado para CPUs ARM
#      (exatamente o que o Raspberry Pi 4/5 usa)
#   3. Gera uma pasta  models/wood_ncnn_model/  com dois arquivos:
#        - wood_ncnn_model.ncnn.param  (arquitetura da rede)
#        - wood_ncnn_model.ncnn.bin    (pesos em float16)
#
# Por que NCNN e não TFLite?
#   - NCNN foi criado pela Tencent especificamente para ARM/Linux
#   - Ultralytics exporta nativamente sem dependências extras
#   - 2-4x mais rápido que rodar o .pt direto na CPU do Pi
#   - Não precisa de GPU — roda puro na CPU do Raspberry
#
# Tempo estimado de export: ~2-5 min no seu PC
# ============================================================
from pathlib import Path
from ultralytics import YOLO

# Usa o melhor modelo treinado
# (Se acabou de treinar, BEST_MODEL_PATH já está definido na Célula 6)
# (Se reiniciou o kernel, rode a Célula 11 primeiro para recarregar)
MODEL_TO_EXPORT = BEST_MODEL_PATH  # ex: runs/wood_yolov8m/weights/best.pt

print(f"📦 Exportando: {MODEL_TO_EXPORT}")
print("⏳ Isso pode demorar alguns minutos...")

model_export = YOLO(str(MODEL_TO_EXPORT))

# Export para NCNN
# half=True  → pesos em float16 (metade do tamanho, mesma precisão prática)
# imgsz=640  → mesmo tamanho do treino
export_path = model_export.export(
    format="ncnn",
    imgsz=640,
    half=True,       # float16 — reduz tamanho ~50% sem perda significativa
    dynamic=False,   # batch fixo = mais rápido no Pi
    simplify=True,   # simplifica o grafo ONNX intermediário
)

print(f"\n✅ Export concluído!")
print(f"📁 Pasta gerada: {export_path}")

# Copia a pasta para models/ para facilitar o envio ao Raspberry
import shutil
NCNN_DEST = MODELS_DIR / "wood_ncnn_model"
if NCNN_DEST.exists():
    shutil.rmtree(NCNN_DEST)
shutil.copytree(export_path, NCNN_DEST)

print(f"💾 Cópia salva em: {NCNN_DEST}")
print()
print("📋 Arquivos gerados:")
for f in sorted(NCNN_DEST.rglob("*")):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name:40s}  {size_kb:8.1f} KB")
print()
print("🚀 Próximo passo: copie a pasta  models/wood_ncnn_model/  para o Raspberry Pi")
print("   Comando exemplo:")
print("   scp -r models/wood_ncnn_model pi@192.168.1.XX:/home/pi/projeto/models/")


# Teste Local do Modelo NCNN (simula o Raspberry Pi)

In [ ]:
# ============================================================
# CÉLULA 12 — Teste do modelo NCNN (simula o Raspberry Pi)
#
# Esta célula testa o modelo NCNN aqui no seu PC para confirmar
# que o export funcionou antes de mandar para o Raspberry.
#
# No Raspberry Pi, o script de produção será o main.py separado
# (Tarefa 2 do projeto), mas esta célula serve para validar.
#
# IMPORTANTE — Para instalar no Raspberry Pi:
#   pip install ultralytics opencv-python-headless
#   (NÃO precisa de torch nem CUDA — o NCNN roda puro em CPU)
# ============================================================
import time
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from pathlib import Path
from ultralytics import YOLO

# ─── Carrega o modelo NCNN (sem GPU, sem torch) ───────────────
NCNN_MODEL_PATH = MODELS_DIR / "wood_ncnn_model"

if not NCNN_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Pasta NCNN não encontrada em {NCNN_MODEL_PATH}.\n"
        "Execute a Célula 10 (Export NCNN) primeiro."
    )

print("📦 Carregando modelo NCNN (sem GPU)...")
# Força device='cpu' — exatamente como vai rodar no Raspberry
model_ncnn = YOLO(str(NCNN_MODEL_PATH), task="detect")
print("✅ Modelo NCNN carregado!")

# ─── Configurações de inferência ──────────────────────────────
CONF_THRESH = 0.40   # Limiar de confiança
IOU_THRESH  = 0.55   # Limiar de NMS

# ─── Seleciona imagens de validação para teste ────────────────
val_imgs = list(VAL_IMG.glob("*"))
sample_imgs = random.sample(val_imgs, min(6, len(val_imgs)))

# ─── Benchmark de velocidade (simula o Pi) ────────────────────
print("\n⏱️  Benchmark de velocidade (CPU only, como no Raspberry Pi):")
tempos = []
for img_path in sample_imgs[:3]:
    t0 = time.perf_counter()
    model_ncnn.predict(
        source=str(img_path),
        conf=CONF_THRESH,
        iou=IOU_THRESH,
        imgsz=640,
        device="cpu",
        verbose=False,
    )
    t1 = time.perf_counter()
    tempos.append(t1 - t0)

media_ms = (sum(tempos) / len(tempos)) * 1000
print(f"   Média por frame : {media_ms:.0f} ms")
print(f"   FPS estimado    : {1000/media_ms:.1f} fps")
print()
if media_ms < 500:
    print("✅ Velocidade OK para o Raspberry Pi 4/5")
else:
    print("⚠️  Pode ser lento no Pi — considere reduzir imgsz para 320")

# ─── Visualização das detecções ───────────────────────────────
print("\n🔍 Testando detecções nas amostras de validação:")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    preds = model_ncnn.predict(
        source=str(img_path),
        conf=CONF_THRESH,
        iou=IOU_THRESH,
        imgsz=640,
        device="cpu",
        verbose=False,
    )
    result = preds[0]
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    n = len(result.boxes)
    status = f"⚠️ {n} defeito(s)" if n > 0 else "✅ Sem defeitos"
    ax.imshow(annotated)
    ax.set_title(f"{img_path.name}\n{status} [NCNN/CPU]", fontsize=8, fontweight="bold")
    ax.axis("off")

plt.suptitle(
    "Detecção NCNN (CPU only) — Simula comportamento no Raspberry Pi",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
out_path = MODELS_DIR / "ncnn_validation_samples.png"
plt.savefig(str(out_path), dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Salvo em: {out_path}")
print()
print("🚀 Se as detecções parecem corretas, o modelo está pronto para o Raspberry Pi!")
print(f"   Copie a pasta: models/wood_ncnn_model/  →  /home/pi/projeto/models/")
